In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
import time

**Generalization and Netowrk Design Strategies**


* Cost Function $$ C = \frac{1}{P}\sum_p \sum_o \frac{1}{2}(D_{op}-X_{op})^2 $$

* A particular weight $u_k$ (that controls several connection strengths).  $$ u_k \xleftarrow{} u_k +\varepsilon_k \sum_{(i,j)\exists V_k}\frac{\delta C}{\delta w_{ij}}$$

$ w_{ij}$ is the connection strength from unit j to i. $V_k$ is the set of unit index pairs (i,j) such that the connection strength $w_{ij}$ is controlled by the weight $u_k$. 
The step size $\varepsilon_k$ is not constant but is a function of the curvature of the cost function along the axis $u_k$. The expression for $\varepsilon_k$ is 
$$ \varepsilon_k=\frac{\lambda}{\mu +h_{kk}}$$
where $\lambda,\mu$ are constants and $h_{kk}$ is the running estimate of the second derivative of the cost function C with respect to $u_k$. 

In [ ]:
from MNIST_import import MnistDataloader

training_images_filepath = r"C:\Users\User\.cache\kagglehub\datasets\hojjatk\mnist-dataset\versions\1\train-images-idx3-ubyte\train-images-idx3-ubyte"
training_labels_filepath = r"C:\Users\User\.cache\kagglehub\datasets\hojjatk\mnist-dataset\versions\1\train-labels-idx1-ubyte\train-labels-idx1-ubyte"
test_labels_filepath     = r"C:\Users\User\.cache\kagglehub\datasets\hojjatk\mnist-dataset\versions\1\t10k-labels-idx1-ubyte\t10k-labels-idx1-ubyte" 
test_images_filepath     = r"C:\Users\User\.cache\kagglehub\datasets\hojjatk\mnist-dataset\versions\1\t10k-images-idx3-ubyte\t10k-images-idx3-ubyte"
dataloader = MnistDataloader(
    training_images_filepath,
    training_labels_filepath,
    test_images_filepath,
    test_labels_filepath,
)
(x_train, y_train), (x_test, y_test) = dataloader.load_data()
# x_train = x_train[: int(len(x_train) * 0.1)]
# y_train = y_train[: int(len(y_train) * 0.1)]
# x_test = x_test[: int(len(x_test) * 0.1)]
# y_test = y_test[: int(len(y_test) * 0.1)]

print("x_train shape:", len(x_train))
print("y_train shape:", len(y_train))
print("x_test shape:", len(x_test))
print("y_test shape:", len(y_test))
print("Image shape:", len(x_train[1][0]), len(x_train[1]))

Notes on determining the code structure for the NN. 
1. Each weight has an index ij from the unit j to i. If the weights for a specific i'th cell are in their own list, it will be easier for python to call the index of that list, and compute the dot product. That product can be taken into a list. Compute the time it takes to pop() and sum each item in the list, vs the blackbox sum() operation for lists.
2. The size of the weight list for each cell will then be the size of the layer before it.
3. There will need to be i lists of weights. 
4. Working with only a few hidden layers, these will not be indexed, but rather named. 
5. Net 1 in this paper contains 2570 weights. There training data vectors were 16x16 pixel maps, or 256 values. 10 hidden units creates 2560 weight values. Each cell has its own bias, making 5270 weights.
6. In this test, we use the MNIST training data, with 28x28 pixels, forming a 784 value column vector, and making 7850 weights.

In [ ]:
print("Content output of x_train shape:",np.shape(x_train))
print(" Content output of length of various inputs:",len(x_train), len(y_train), len(x_test), len(y_test))

The training solution set $d_{op}$ lies within the y_train[] list. To create a solution vector, start with a vector of ten 0's, initialized from 0-9. We utilize the value of the solution as the index, and change the value of the vector at that index to 1.

In [ ]:
import time
import numpy as np

start = time.time()
d = [[1 if j == val else 0 for j in range(10)] for val in y_train]
#print("Length of d (number of solutions):", len(d), "Ex: Solution:" + str(d[3]) + ". Compare solution to y_train[3]: " + str(y_train[3]))
end = time.time()
print("Time taken to create d:", end - start)

start = time.time()
d = []
for i in range(len(y_train)):
    c = ([0]*10)
    c[y_train[i]] = 1
    d.append(c)
#print("Length of d (number of solutions):", len(d), "Ex: Solution:" + str(d[3]) + ". Compare solution to y_train[3]: " + str(y_train[3]))
end = time.time()
print("Time taken to create d:", end - start)

In [44]:
# Initialize size of first hidden layer of the neural network

layer_1_size = 784
output_layer_size = 10
numLayers = 1 
total_weights_layer_1 = layer_1_size*output_layer_size  + output_layer_size  # Total number of weights in the first hidden layer
# Find a method for initializing the weights of the neural network. Bounds, based on Le Cun (1986) [-2.4/F, 2.4/F] or [-2.4/sqrt(F), 2.4/sqrt(F)]
layer_1_weight_range = 2.4 / layer_1_size  # Le Cun (1986) weight initialization bounds
input2output_weights = np.random.uniform(-layer_1_weight_range, layer_1_weight_range, size=(numLayers, output_layer_size, layer_1_size))  # Initialize weights for the first hidden layer

# Variables for the scaled hyperbolic tangent function
A = 1.7159
B = 2/3

print("Total number of weights in the first hidden layer:", total_weights_layer_1)
print("First weights index size:" , len(input2output_weights))
print("Second weights index size:" , len(input2output_weights[0]))
print("Last weights index size:", len(input2output_weights[0][0][:]))

def dot(a, b): # Returns the dot product of two vectors a and b. a and b must be the same length.
    if len(a) != len(b):
        raise ValueError("Vectors must be the same length")
    return sum(x * y for x, y in zip(a, b))

def image2vector(image): # Converts a 2D image into a 1D column vector
    c = []
    for row in range(len(image)):
        for col in range(len(image[row])):
            c.append(image[row][col])
    return c

def vector_difference_RMS(a, b): # Returns the sum of the differences between two vectors a and b. a and b must be the same length.
    if len(a) != len(b):
        raise ValueError("Vectors must be the same length")
    return sum(abs(x - y)**2 for x, y in zip(a, b))


# Compute the number of dot product between the input and the weights of the first hidden layer for each neuron. Time this process to understand how scalable the neural network is.

# Pass each of the neuron dot products through the scaled hyperbolic tangent function to get the output of the first hidden layer.
# Time this process to understand how scalable the neural network is.

print(np.shape(x_train)[1])
print(np.shape(x_train)[0])
print(y_train[2])
print(np.shape(y_train)[0])

Total number of weights in the first hidden layer: 7850
First weights index size: 1
Second weights index size: 10
Last weights index size: 784
28
60000
4
60000


**Computing the gradient of the cost function**

To compute $ \frac{C}{w_ij}$, which is how the overall test cost function changes with each weight connection, we use the chain rule. 

e.g. C(output_vector_cell(dot_product(weight_ij)))

We then find 

$$ \frac{\partial C}{\partial w_{ij}} = \frac{\partial C}{\partial X_{j}}\frac{\partial X_j}{\partial a_{j}}\frac{\partial a_j}{\partial w_{ij}}$$

Where:
$$\frac{\partial C}{\partial X_{j}} = (X_j-D_j)$$

Here, $D_i$ is the solution vector.

$$\frac{\partial X_j}{\partial a_j} = SA \operatorname{sech}^2 (Sa)$$

Where $S$ and $A$ are constants, and $a$ is the dot product of the input vector $u$ and the weight vector $w_i$. The variable $w_i$ describes a row vector of weights connecting all input units $i$ to output unit $j$.

$$\frac{\partial a_j}{\partial w_{ij}} = u_{ij}$$

As a result, $\frac{\partial C}{\partial w_{ij}} = (X_i-D_i)SA u_{ij}\operatorname{sech}^2 (Sa)$

From this derivation, we have a few options. $u_{ij}$ shows that the gradient descent is a function of the input vector. We can then update the weights after every input vector, called a _stochastic gradient descent_, we can sum the weight change decisions of each vector, and either use that sum, or the average, and this is known as a _batch gradient descent_, or we can sum sum n number of weight changes, and update every n changes, known as _mini-batch gradient descent_. SGD is faster, but only works if the training data is very repetetive, which it is in this case. For the purposes of learning, all 3 will be attempted.

**Stochastic Gradient Descent**

In [ ]:
start = time.time()
error = 0
for c in range(len(x_train)):
    # For each datapoint in the training set, compute the dot prdouct between the input image and the weights of the first hidden layer, 
    # and then pass the result through the scaled hyperbolic tangent function to get the output of the first hidden layer. 
    output_layer = A*[math.tanh(B * dot(image2vector(x_train[c]), input2output_weights[0][i])) for i in range(output_layer_size)]  # Compute the output of the first hidden layer for the first training image
    
    error += vector_difference_RMS(output_layer, d[c])  # Compute the error between the output of the first hidden layer and the expected output for the first training image
error = error / (2 * len(x_train))  # Compute the average error over all training images
# checkLater: need to change c to the range of the length of x_train to compute the output for all training images, but this will take a long time to compute. For now, we will just compute the output for the first 10 training images.
# about 9.5 ms per dot product, so about 9.5 min for 10,000 images.
end = time.time()
print("Time taken to compute output_layer:", end - start)
print("output_layer dimensions:", len(output_layer))
print("output_layer[0]:", output_layer[0])